In [1]:
import os
import re
import shutil
from pathlib import Path
from collections import Counter
from typing import List, Tuple, Dict

import pandas as pd
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_community.document_loaders import PyMuPDFLoader, DirectoryLoader
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

c:\Users\Lenovo\Desktop\Resume+DJ match maker\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ===== CHANGE THIS =====
RESUME_DIR = r"C:/Users/Lenovo/Desktop/Resume+DJ match maker/docc"
CHROMA_DIR = "./chroma_resume_optimized"

# Better quality, but more cost
EMBED_MODEL = "text-embedding-3-large"

# Cheaper option:
# EMBED_MODEL = "text-embedding-3-small"

TOP_K_FINAL = 5
FETCH_K = 20

JOB_DESCRIPTION_TEXT = '''Data Scientist Intern – Microsoft

At Microsoft, Data Scientist Interns work on real product problems using data, machine learning, and experimentation. You will collaborate with engineers, product managers, and researchers to analyze data, build models, and generate insights that improve Microsoft products and services used by millions of people worldwide.

During the internship, you will explore real datasets, identify patterns, and apply statistical and machine learning techniques to solve business and product challenges. You will learn how data science decisions are made in production environments and how analytical insights translate into real product improvements.

You will also communicate your findings to cross-functional teams, helping stakeholders understand what the data reveals and how it can guide product decisions.

Microsoft internships are designed to give students hands-on experience working on meaningful problems while learning from experienced engineers and data scientists across global teams.

Responsibilities

Apply statistical analysis, machine learning, and data exploration techniques to solve real product problems

Work with structured and unstructured data to uncover patterns, insights, and opportunities

Design and analyze experiments to evaluate product changes or new features

Interpret results and translate them into clear recommendations for product and engineering teams

Collaborate with engineers, product managers, and other data scientists to improve Microsoft products

Present findings and insights through reports, dashboards, or visualizations

Qualifications

Currently pursuing a Bachelor’s or Master’s degree in Data Science, Computer Science, Statistics, Mathematics, Economics, or a related quantitative field

Experience with Python, R, or similar programming languages for data analysis

Familiarity with statistics, machine learning concepts, and data analysis techniques

Strong problem-solving skills and the ability to communicate analytical findings clearly

Must have at least one semester/quarter remaining in the degree program after the internship'''
 

In [3]:
load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found. Add it in your .env file first.")

print("OPENAI_API_KEY loaded successfully")

OPENAI_API_KEY loaded successfully


In [4]:
HEADINGS = {
    "role overview",
    "core responsibilities",
    "key areas of specialization",
    "minimum qualifications",
    "preferred qualifications",
}

STOPWORDS = {
    "a", "an", "the", "and", "or", "to", "of", "for", "in", "on", "with", "using",
    "by", "at", "from", "into", "across", "is", "are", "be", "as", "that", "this",
    "will", "can", "have", "has", "had", "it", "its", "their", "your", "you",
    "our", "we", "they", "them", "who", "what", "when", "where", "why", "how",
    "experience", "preferred", "minimum", "qualification", "qualifications",
    "responsibilities", "role", "overview", "development", "building", "maintaining"
}

BAD_HINTS = {
    "linkedin", "leetcode", "hackerrank", "codechef", "portfolio", "references",
    "hobbies", "interests", "address", "phone", "email"
}

IMPORTANT_TERMS = {
    "llm", "rag", "transformer", "nlp", "computer vision", "mlops", "deployment",
    "evaluation", "monitoring", "quantization", "distillation", "pruning", "android",
    "distributed", "pytorch", "tensorflow", "python", "production", "inference",
    "fine-tuning", "guardrails", "classifier", "chatbot", "voicebot"
}

def normalize_text(text: str) -> str:
    text = text.replace("\u2022", "•")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def normalize_for_tokens(text: str) -> str:
    text = text.lower().replace("ml", "machine learning").replace("ai", "artificial intelligence")
    text = re.sub(r"[^a-z0-9+#./ -]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize(text: str) -> List[str]:
    return [t for t in normalize_for_tokens(text).split() if len(t) > 2 and t not in STOPWORDS]

def unique_keep_order(items: List[str]) -> List[str]:
    seen = set()
    out = []
    for item in items:
        key = item.strip().lower()
        if not key or key in seen:
            continue
        seen.add(key)
        out.append(item.strip())
    return out

def snippet(text: str, n: int = 260) -> str:
    text = normalize_text(text)
    return text[:n] + ("..." if len(text) > n else "")

def keyword_overlap_score(query: str, text: str) -> float:
    q = set(tokenize(query))
    t = set(tokenize(text))
    if not q or not t:
        return 0.0
    overlap = len(q & t)
    return overlap / max(len(q), 1)

def phrase_bonus(query: str, text: str) -> float:
    qn = normalize_for_tokens(query)
    tn = normalize_for_tokens(text)
    bonus = 0.0

    for phrase in IMPORTANT_TERMS:
        if phrase in qn and phrase in tn:
            bonus += 0.12

    if "production" in qn and any(w in tn for w in ["deployed", "deployment", "production", "serving"]):
        bonus += 0.18

    if "evaluation" in qn and any(w in tn for w in ["evaluation", "metrics", "accuracy", "f1", "precision", "recall"]):
        bonus += 0.12

    return min(bonus, 0.5)

def bad_chunk_penalty(text: str) -> float:
    low = normalize_for_tokens(text)
    penalty = 0.0

    if any(h in low for h in BAD_HINTS):
        penalty += 0.2

    if len(low.split()) < 18:
        penalty += 0.15

    return penalty

def section_bonus(doc: Document, query: str) -> float:
    section = str(doc.metadata.get("section", "")).lower()
    q = normalize_for_tokens(query)
    bonus = 0.0

    if any(s in section for s in ["experience", "projects", "work experience"]):
        bonus += 0.1

    if "skills" in section and any(term in q for term in ["python", "tensorflow", "pytorch", "rag", "llm"]):
        bonus += 0.08

    return bonus

In [5]:
def load_resume_documents(resume_dir: str) -> List[Document]:
    dir_loader = DirectoryLoader(
        resume_dir,
        glob="**/*.pdf",
        loader_cls=PyMuPDFLoader,
        show_progress=True,
    )
    docs = dir_loader.load()

    for d in docs:
        d.page_content = d.page_content.replace("\x00", " ")
        d.metadata["source_file"] = Path(d.metadata.get("source", "unknown")).name

    return docs

In [6]:
def detect_section(line: str) -> str | None:
    line_clean = normalize_for_tokens(line)

    section_map = {
        "summary": ["summary", "profile", "objective"],
        "experience": ["experience", "work experience", "employment"],
        "projects": ["projects", "academic projects", "personal projects"],
        "education": ["education"],
        "skills": ["skills", "technical skills", "tech stack", "technologies"],
        "certifications": ["certifications", "certificates"],
    }

    for section, aliases in section_map.items():
        if any(alias in line_clean for alias in aliases) and len(line_clean.split()) <= 5:
            return section
    return None


def pdf_pages_to_semantic_blocks(pdf_documents: List[Document]) -> List[Document]:
    blocks: List[Document] = []

    for page_doc in pdf_documents:
        source_file = page_doc.metadata.get("source_file", "unknown")
        page_num = page_doc.metadata.get("page", 0)
        current_section = "unknown"

        lines = [ln.strip() for ln in page_doc.page_content.splitlines()]
        lines = [ln for ln in lines if ln]

        current_block: List[str] = []

        for line in lines:
            maybe_section = detect_section(line)

            if maybe_section:
                if current_block:
                    blocks.append(
                        Document(
                            page_content="\n".join(current_block),
                            metadata={
                                "source_file": source_file,
                                "page": page_num,
                                "section": current_section,
                            },
                        )
                    )
                    current_block = []

                current_section = maybe_section
                continue

            bulletish = bool(re.match(r"^(•|-|\*|\d+[.)])\s+", line))
            current_block.append(line)

            joined = "\n".join(current_block)

            if len(joined) > 900 or (bulletish and len(current_block) >= 4):
                blocks.append(
                    Document(
                        page_content=joined,
                        metadata={
                            "source_file": source_file,
                            "page": page_num,
                            "section": current_section,
                        },
                    )
                )
                current_block = []

        if current_block:
            blocks.append(
                Document(
                    page_content="\n".join(current_block),
                    metadata={
                        "source_file": source_file,
                        "page": page_num,
                        "section": current_section,
                    },
                )
            )

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1100,
        chunk_overlap=180,
        separators=["\n\n", "\n•", "\n-", "\n", ". ", " ", ""],
    )

    refined = splitter.split_documents(blocks)

    for i, d in enumerate(refined):
        d.metadata["chunk_id"] = i
        d.page_content = normalize_text(d.page_content)

    return refined

In [7]:
def split_requirement_line(line: str) -> List[str]:
    line = line.strip("•- \t")
    if not line:
        return []

    if ":" in line:
        left, right = line.split(":", 1)
        if len(left.split()) <= 5 and right.strip():
            return [
                f"{left.strip()}: {part.strip()}"
                for part in re.split(r";|\. ", right)
                if part.strip()
            ]

    parts = [p.strip() for p in re.split(r";|\. ", line) if len(p.strip()) >= 12]
    return parts or [line]


def jd_to_requirement_docs(jd_text: str) -> List[Document]:
    heading = "general"
    requirements: List[Document] = []

    for raw in jd_text.splitlines():
        line = raw.strip()
        if not line:
            continue

        low = line.lower().strip()
        if low in HEADINGS:
            heading = line.strip()
            continue

        if len(line) < 12:
            continue

        for req in split_requirement_line(line):
            req = req.strip()
            if len(req) < 12:
                continue

            requirements.append(
                Document(
                    page_content=f"{heading} — {req}",
                    metadata={"source": "jd", "heading": heading},
                )
            )

    deduped = []
    seen = set()

    for d in requirements:
        key = normalize_for_tokens(d.page_content)
        if key in seen:
            continue
        seen.add(key)
        deduped.append(d)

    return deduped

In [8]:
def build_vectorstore(resume_chunks: List[Document]):
    shutil.rmtree(CHROMA_DIR, ignore_errors=True)

    embeddings = OpenAIEmbeddings(model=EMBED_MODEL)

    vectorstore = Chroma.from_documents(
        documents=resume_chunks,
        embedding=embeddings,
        collection_name="resume_chunks_optimized",
        persist_directory=CHROMA_DIR,
    )

    return vectorstore

In [9]:
def expand_query(jd_requirement: str) -> List[str]:
    base = jd_requirement.strip()
    variants = [base]

    norm = normalize_for_tokens(base)
    terms = []

    for phrase in IMPORTANT_TERMS:
        if phrase in norm:
            terms.append(phrase)

    token_counts = Counter(tokenize(base))
    strong_terms = [t for t, _ in token_counts.most_common(8)]

    if strong_terms:
        variants.append("Relevant resume evidence for: " + " ".join(strong_terms))

    if terms:
        variants.append(base + " Focus on " + ", ".join(terms))

    variants.append(base + " Include concrete project, deployment, tools, metrics, and production evidence")

    return unique_keep_order(variants)


def retrieve_candidates(vectorstore: Chroma, query: str, fetch_k: int = FETCH_K) -> List[Tuple[Document, float]]:
    all_hits: List[Tuple[Document, float]] = []
    seen = set()

    for q in expand_query(query):
        hits = vectorstore.similarity_search_with_score(q, k=fetch_k)

        for doc, score in hits:
            key = (
                doc.metadata.get("source_file"),
                doc.metadata.get("page"),
                doc.metadata.get("chunk_id"),
            )
            if key in seen:
                continue
            seen.add(key)
            all_hits.append((doc, float(score)))

    return all_hits


def rerank_candidates(jd_requirement: str, candidates: List[Tuple[Document, float]], top_k: int = TOP_K_FINAL):
    ranked = []

    for doc, vec_score in candidates:
        text = doc.page_content

        overlap = keyword_overlap_score(jd_requirement, text)
        phrase = phrase_bonus(jd_requirement, text)
        sec_bonus = section_bonus(doc, jd_requirement)
        penalty = bad_chunk_penalty(text)

        vec_sim = 1 / (1 + max(vec_score, 0.0))
        final_score = (0.55 * vec_sim) + (0.30 * overlap) + phrase + sec_bonus - penalty

        ranked.append(
            {
                "doc": doc,
                "vector_distance": round(vec_score, 4),
                "vector_similarity": round(vec_sim, 4),
                "keyword_overlap": round(overlap, 4),
                "final_score": round(final_score, 4),
            }
        )

    ranked.sort(key=lambda x: x["final_score"], reverse=True)

    final = []
    used_keys = set()

    for item in ranked:
        doc = item["doc"]
        key = (doc.metadata.get("page"), doc.metadata.get("chunk_id"))

        if key in used_keys:
            continue

        used_keys.add(key)
        final.append(item)

        if len(final) >= top_k:
            break

    return final

In [10]:
def run_pipeline(resume_dir: str, jd_text: str):
    pdf_documents = load_resume_documents(resume_dir)
    if not pdf_documents:
        raise ValueError("No PDF resume found in the given folder.")

    resume_chunks = pdf_pages_to_semantic_blocks(pdf_documents)
    jd_requirements = jd_to_requirement_docs(jd_text)
    vectorstore = build_vectorstore(resume_chunks)

    results = []

    for i, jd_doc in enumerate(jd_requirements):
        jd_req = jd_doc.page_content
        candidates = retrieve_candidates(vectorstore, jd_req, fetch_k=FETCH_K)
        ranked = rerank_candidates(jd_req, candidates, top_k=TOP_K_FINAL)

        results.append(
            {
                "jd_req_id": i,
                "jd_requirement": jd_req,
                "matches": ranked,
            }
        )

    return pdf_documents, resume_chunks, jd_requirements, results

In [11]:
def build_summary_df(results: List[Dict]) -> pd.DataFrame:
    rows = []

    for item in results:
        jd_id = item["jd_req_id"]
        jd_req = item["jd_requirement"]
        matches = item["matches"]

        if not matches:
            rows.append(
                {
                    "jd_req_id": jd_id,
                    "jd_requirement": jd_req,
                    "best_final_score": None,
                    "resume_page": None,
                    "resume_section": None,
                    "source_file": None,
                    "evidence": None,
                }
            )
            continue

        best = matches[0]
        doc = best["doc"]

        rows.append(
            {
                "jd_req_id": jd_id,
                "jd_requirement": jd_req,
                "best_final_score": best["final_score"],
                "resume_page": doc.metadata.get("page"),
                "resume_section": doc.metadata.get("section"),
                "source_file": doc.metadata.get("source_file"),
                "evidence": snippet(doc.page_content, 320),
            }
        )

    return pd.DataFrame(rows).sort_values("best_final_score", ascending=False)


def build_detailed_df(results: List[Dict]) -> pd.DataFrame:
    rows = []

    for item in results:
        for rank, match in enumerate(item["matches"], start=1):
            doc = match["doc"]

            rows.append(
                {
                    "jd_req_id": item["jd_req_id"],
                    "rank": rank,
                    "jd_requirement": item["jd_requirement"],
                    "final_score": match["final_score"],
                    "vector_distance": match["vector_distance"],
                    "keyword_overlap": match["keyword_overlap"],
                    "resume_page": doc.metadata.get("page"),
                    "resume_section": doc.metadata.get("section"),
                    "source_file": doc.metadata.get("source_file"),
                    "chunk_id": doc.metadata.get("chunk_id"),
                    "evidence": snippet(doc.page_content, 450),
                }
            )

    return pd.DataFrame(rows).sort_values(["jd_req_id", "rank"])

In [12]:
pdf_documents, resume_chunks, jd_requirements, results = run_pipeline(
    resume_dir=RESUME_DIR,
    jd_text=JOB_DESCRIPTION_TEXT,
)

print(f"Loaded pages: {len(pdf_documents)}")
print(f"Resume chunks: {len(resume_chunks)}")
print(f"JD requirements: {len(jd_requirements)}")

summary_df = build_summary_df(results)
detailed_df = build_detailed_df(results)

print("\n===== SUMMARY =====")
display(summary_df.head(15))

print("\n===== DETAILED =====")
display(detailed_df.head(20))

100%|██████████| 1/1 [00:01<00:00,  1.05s/it]


Loaded pages: 1
Resume chunks: 8
JD requirements: 20

===== SUMMARY =====


,jd_req_id,jd_requirement,best_final_score,resume_page,resume_section,source_file,evidence
16,16,"general — Experience with Python, R, or simila...",0.6344,0,skills,Devanshi_Faldu_s_Resume (2).pdf,TECHNICALS | Power BI | Data Visualisation | D...
17,17,"general — Familiarity with statistics, machine...",0.4716,0,experience,Devanshi_Faldu_s_Resume (2).pdf,AI INTERN Nexeo Security September 2024 - Marc...
0,0,general — Data Scientist Intern – Microsoft,0.4637,0,experience,Devanshi_Faldu_s_Resume (2).pdf,AI INTERN Nexeo Security September 2024 - Marc...
8,8,"general — Apply statistical analysis, machine ...",0.4392,0,experience,Devanshi_Faldu_s_Resume (2).pdf,AI INTERN Nexeo Security September 2024 - Marc...
1,1,"general — At Microsoft, Data Scientist Interns...",0.4116,0,experience,Devanshi_Faldu_s_Resume (2).pdf,AI INTERN Nexeo Security September 2024 - Marc...
10,10,general — Design and analyze experiments to ev...,0.3821,0,projects,Devanshi_Faldu_s_Resume (2).pdf,MOVIE RECOMMENDATION | Similar movie suggestio...
3,3,"general — During the internship, you will expl...",0.3805,0,experience,Devanshi_Faldu_s_Resume (2).pdf,AI INTERN Nexeo Security September 2024 - Marc...
9,9,general — Work with structured and unstructure...,0.3750,0,experience,Devanshi_Faldu_s_Resume (2).pdf,AI INTERN Nexeo Security September 2024 - Marc...
2,2,"general — You will collaborate with engineers,...",0.3700,0,experience,Devanshi_Faldu_s_Resume (2).pdf,AI INTERN Nexeo Security September 2024 - Marc...
11,11,general — Interpret results and translate them...,0.3648,0,projects,Devanshi_Faldu_s_Resume (2).pdf,MOVIE RECOMMENDATION | Similar movie suggestio...



===== DETAILED =====


,jd_req_id,rank,jd_requirement,final_score,vector_distance,keyword_overlap,resume_page,resume_section,source_file,chunk_id,evidence
0,0,1,general — Data Scientist Intern – Microsoft,0.4637,1.2571,0.4000,0,experience,Devanshi_Faldu_s_Resume (2).pdf,2,AI INTERN Nexeo Security September 2024 - Marc...
1,0,2,general — Data Scientist Intern – Microsoft,0.3780,1.5229,0.2000,0,projects,Devanshi_Faldu_s_Resume (2).pdf,3,MOVIE RECOMMENDATION | Similar movie suggestio...
2,0,3,general — Data Scientist Intern – Microsoft,0.3171,1.1394,0.2000,0,skills,Devanshi_Faldu_s_Resume (2).pdf,5,TECHNICALS | Power BI | Data Visualisation | D...
3,0,4,general — Data Scientist Intern – Microsoft,0.3067,1.2294,0.2000,0,education,Devanshi_Faldu_s_Resume (2).pdf,1,DAYANAND SAGAR UNIVERSITY | Masters of Science...
4,0,5,general — Data Scientist Intern – Microsoft,0.1614,1.6020,0.0000,0,experience,Devanshi_Faldu_s_Resume (2).pdf,4,•Implement 3 combined feature vectors to compu...
5,1,1,"general — At Microsoft, Data Scientist Interns...",0.4116,1.3244,0.2500,0,experience,Devanshi_Faldu_s_Resume (2).pdf,2,AI INTERN Nexeo Security September 2024 - Marc...
6,1,2,"general — At Microsoft, Data Scientist Interns...",0.3380,1.5821,0.0833,0,projects,Devanshi_Faldu_s_Resume (2).pdf,3,MOVIE RECOMMENDATION | Similar movie suggestio...
7,1,3,"general — At Microsoft, Data Scientist Interns...",0.2649,1.2926,0.0833,0,skills,Devanshi_Faldu_s_Resume (2).pdf,5,TECHNICALS | Power BI | Data Visualisation | D...
8,1,4,"general — At Microsoft, Data Scientist Interns...",0.2584,1.3569,0.0833,0,education,Devanshi_Faldu_s_Resume (2).pdf,1,DAYANAND SAGAR UNIVERSITY | Masters of Science...
9,1,5,"general — At Microsoft, Data Scientist Interns...",0.1571,1.6560,0.0000,0,experience,Devanshi_Faldu_s_Resume (2).pdf,4,•Implement 3 combined feature vectors to compu...


In [14]:
pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]

Note: you may need to restart the kernel 

In [15]:
summary_df.to_csv("optimized_summary_matches.csv", index=False)
detailed_df.to_csv("optimized_detailed_matches.csv", index=False)

summary_df.to_excel("optimized_summary_matches.xlsx", index=False)
detailed_df.to_excel("optimized_detailed_matches.xlsx", index=False)

print("Saved:")
print("1. optimized_summary_matches.csv")
print("2. optimized_detailed_matches.csv")
print("3. optimized_summary_matches.xlsx")
print("4. optimized_detailed_matches.xlsx")

Saved:
1. optimized_summary_matches.csv
2. optimized_detailed_matches.csv
3. optimized_summary_matches.xlsx
4. optimized_detailed_matches.xlsx


In [18]:
req_index = 0

print("JD REQUIREMENT:\n")
print(results[req_index]["jd_requirement"])

print("\nTOP MATCHES:\n")
for i, match in enumerate(results[req_index]["matches"], start=1):
    doc = match["doc"]
    print(f"Rank: {i}")
    print(f"Final Score: {match['final_score']}")
    print(f"Page: {doc.metadata.get('page')}")
    print(f"Section: {doc.metadata.get('section')}")
    print(f"Source File: {doc.metadata.get('source_file')}")
    print("Evidence:")
    print(snippet(doc.page_content, 500))
    print("-" * 80)

JD REQUIREMENT:

general — Data Scientist Intern – Microsoft

TOP MATCHES:

Rank: 1
Final Score: 0.4637
Page: 0
Section: experience
Source File: Devanshi_Faldu_s_Resume (2).pdf
Evidence:
AI INTERN Nexeo Security September 2024 - March 2025 | Remote •Developed and optimized AI-driven threat detection models that improved anomaly identification accuracy by over 25%, enhancing the organization’s proactive cybersecurity measures. •Automated security data analysis workflows using Python and machine learning algorithms, reducing manual review time by 40% and improving incident response efficiency. •Implemented NLP-based log analysis tools to detect phishing and malware indicators, inc...
--------------------------------------------------------------------------------
Rank: 2
Final Score: 0.378
Page: 0
Section: projects
Source File: Devanshi_Faldu_s_Resume (2).pdf
Evidence:
MOVIE RECOMMENDATION | Similar movie suggestion using Streamlit Python | Streamlit | Pandas | NumPy | scikit-learn | NLT

# upgrading the app

In [19]:
import json
from openai import OpenAI

client = OpenAI()

In [20]:
def extract_json_block(text: str):
    text = text.strip()

    if text.startswith("```json"):
        text = text.replace("```json", "", 1).strip()
    if text.startswith("```"):
        text = text.replace("```", "", 1).strip()
    if text.endswith("```"):
        text = text[:-3].strip()

    start = text.find("{")
    end = text.rfind("}")

    if start != -1 and end != -1 and end > start:
        text = text[start:end + 1]

    return json.loads(text)

In [21]:
def classify_jd_requirement(requirement_text: str) -> dict:
    prompt = f"""
You are an expert resume and job-description analyst.

Analyze this job requirement and return STRICT JSON.

Requirement:
{requirement_text}

Return JSON with this exact schema:
{{
  "requirement_text": "...",
  "category": "skills|tools|years|domain|responsibility|other",
  "skills": [],
  "tools": [],
  "years_required": "",
  "domain": "",
  "action_verbs": [],
  "seniority_hint": "",
  "must_have": true,
  "short_label": ""
}}

Rules:
- Output only JSON
- "skills" should include concepts like machine learning, NLP, retrieval, evaluation
- "tools" should include technologies like Python, PyTorch, TensorFlow, SQL, AWS
- "years_required" should be a short string like "2+ years" if present, else ""
- "domain" should be short, like "LLM systems", "computer vision", "backend", "AI products"
- "short_label" should be concise
"""

    response = client.responses.create(
        model="gpt-5-mini",
        input=prompt
    )

    text = response.output_text
    return extract_json_block(text)

In [22]:
def build_structured_jd_requirements(jd_requirements: List[Document]) -> pd.DataFrame:
    structured_rows = []

    for i, jd_doc in enumerate(jd_requirements):
        try:
            parsed = classify_jd_requirement(jd_doc.page_content)
        except Exception as e:
            parsed = {
                "requirement_text": jd_doc.page_content,
                "category": "other",
                "skills": [],
                "tools": [],
                "years_required": "",
                "domain": "",
                "action_verbs": [],
                "seniority_hint": "",
                "must_have": True,
                "short_label": f"Requirement {i}"
            }

        parsed["jd_req_id"] = i
        structured_rows.append(parsed)

    return pd.DataFrame(structured_rows)

In [23]:
def collect_resume_evidence_text(matches: List[dict], top_n: int = 3) -> str:
    parts = []

    for match in matches[:top_n]:
        doc = match["doc"]
        section = doc.metadata.get("section", "")
        page = doc.metadata.get("page", "")
        text = normalize_text(doc.page_content)
        parts.append(f"[Section: {section} | Page: {page}] {text}")

    return "\n\n".join(parts)

In [24]:
def get_resume_global_text(resume_chunks: List[Document]) -> str:
    return "\n".join([normalize_text(d.page_content) for d in resume_chunks])


def find_missing_items(structured_jd_df: pd.DataFrame, resume_chunks: List[Document]) -> pd.DataFrame:
    resume_text = normalize_for_tokens(get_resume_global_text(resume_chunks))

    rows = []

    for _, row in structured_jd_df.iterrows():
        skills = row.get("skills", []) or []
        tools = row.get("tools", []) or []

        missing_skills = []
        missing_tools = []

        for skill in skills:
            if normalize_for_tokens(skill) not in resume_text:
                missing_skills.append(skill)

        for tool in tools:
            if normalize_for_tokens(tool) not in resume_text:
                missing_tools.append(tool)

        rows.append({
            "jd_req_id": row["jd_req_id"],
            "short_label": row.get("short_label", ""),
            "category": row.get("category", ""),
            "skills": ", ".join(skills),
            "tools": ", ".join(tools),
            "missing_skills": ", ".join(missing_skills),
            "missing_tools": ", ".join(missing_tools),
            "years_required": row.get("years_required", ""),
            "domain": row.get("domain", "")
        })

    return pd.DataFrame(rows)

In [25]:
def rewrite_resume_bullets_for_requirement(requirement_text: str, matched_evidence: str) -> dict:
    prompt = f"""
You are an expert resume writer.

Your task:
Rewrite resume bullets so they align better with the job requirement, but ONLY use evidence present in the provided resume evidence.
Do not invent achievements, tools, metrics, or experience.
If evidence is weak, make the bullet conservative and truthful.

JOB REQUIREMENT:
{requirement_text}

MATCHED RESUME EVIDENCE:
{matched_evidence}

Return STRICT JSON:
{{
  "requirement_text": "...",
  "resume_strength": "strong|medium|weak",
  "why": "...",
  "rewritten_bullets": [
    "...",
    "...",
    "..."
  ]
}}

Rules:
- Output only JSON
- Max 3 bullets
- Bullets should be ATS-friendly
- Start bullets with strong action verbs
- Do not fabricate numbers
- Keep each bullet concise and professional
"""

    response = client.responses.create(
        model="gpt-5-mini",
        input=prompt
    )

    text = response.output_text
    return extract_json_block(text)

In [26]:
def generate_rewrite_suggestions(results: List[Dict]) -> pd.DataFrame:
    rows = []

    for item in results:
        jd_req_id = item["jd_req_id"]
        jd_requirement = item["jd_requirement"]
        matches = item["matches"]

        evidence_text = collect_resume_evidence_text(matches, top_n=3)

        if not evidence_text.strip():
            rows.append({
                "jd_req_id": jd_req_id,
                "jd_requirement": jd_requirement,
                "resume_strength": "weak",
                "why": "No strong supporting evidence retrieved from resume.",
                "rewritten_bullets": ""
            })
            continue

        try:
            rewritten = rewrite_resume_bullets_for_requirement(jd_requirement, evidence_text)
            bullets = rewritten.get("rewritten_bullets", [])
            bullet_text = "\n".join([f"• {b}" for b in bullets])

            rows.append({
                "jd_req_id": jd_req_id,
                "jd_requirement": jd_requirement,
                "resume_strength": rewritten.get("resume_strength", ""),
                "why": rewritten.get("why", ""),
                "rewritten_bullets": bullet_text
            })
        except Exception as e:
            rows.append({
                "jd_req_id": jd_req_id,
                "jd_requirement": jd_requirement,
                "resume_strength": "unknown",
                "why": f"Rewrite failed: {str(e)}",
                "rewritten_bullets": ""
            })

    return pd.DataFrame(rows)

In [27]:
structured_jd_df = build_structured_jd_requirements(jd_requirements)
missing_items_df = find_missing_items(structured_jd_df, resume_chunks)
rewrite_df = generate_rewrite_suggestions(results)

print("===== STRUCTURED JD =====")
display(structured_jd_df.head(10))

print("===== MISSING ITEMS =====")
display(missing_items_df.head(10))

print("===== REWRITE SUGGESTIONS =====")
display(rewrite_df.head(10))

===== STRUCTURED JD =====


,requirement_text,category,skills,tools,years_required,domain,action_verbs,seniority_hint,must_have,short_label,jd_req_id
0,general — Data Scientist Intern – Microsoft,other,[],[],,data science,[],intern,True,Data Scientist Intern,0
1,"general — At Microsoft, Data Scientist Interns...",responsibility,"[machine learning, data analysis, experimentat...","[Python, SQL, PyTorch, TensorFlow, Azure]",,AI products,"[work on, use, apply, conduct experimentation,...",Intern,True,Data Scientist Intern — ML & experimentation,1
2,"general — You will collaborate with engineers,...",responsibility,"[collaboration, data analysis, machine learnin...","[Python, SQL, PyTorch, TensorFlow, Azure, Powe...",,AI products,"[collaborate, analyze, build, generate, improve]",,True,Product-focused Data Scientist / ML Engineer,2
3,"general — During the internship, you will expl...",responsibility,"[data exploration, statistical analysis, machi...","[Python, SQL, pandas, scikit-learn, Jupyter]",,AI products,"[explore, identify, apply, solve]",intern/junior,True,ML/Data Exploration Internship,3
4,general — You will learn how data science deci...,responsibility,"[data science, evaluation, data-driven decisio...","[Python, SQL, Jupyter, Git]",,product analytics,"[learn, make decisions, translate insights]",entry-level,True,Learn production data science and product anal...,4
5,general — You will also communicate your findi...,responsibility,"[communication, data analysis, data visualizat...","[Python, SQL, Tableau, Looker, Jupyter, AWS, P...",,AI products,"[communicate, present, translate, collaborate,...",mid-level,True,Communicate data insights to product stakeholders,5
6,general — Microsoft internships are designed t...,responsibility,[],[],,general internship,"[give, work, learn]",intern / student / entry-level,True,Microsoft internship (general),6
7,general — Responsibilities,responsibility,[],[],,,[],,True,General responsibilities,7
8,"general — Apply statistical analysis, machine ...",responsibility,"[statistical analysis, machine learning, data ...","[Python, SQL, pandas, scikit-learn, PyTorch, T...",,AI products,"[Apply, analyze, explore, solve]",,True,ML & statistical analysis for product problems,8
9,general — Work with structured and unstructure...,responsibility,"[data analysis, exploratory data analysis, sta...","[Python, SQL, Pandas, Jupyter, Spark, AWS, Ten...",,data analytics,"[work with, analyze, explore, uncover, identif...",entry to mid-level,True,Analyze structured & unstructured data,9


===== MISSING ITEMS =====


,jd_req_id,short_label,category,skills,tools,missing_skills,missing_tools,years_required,domain
0,0,Data Scientist Intern,other,,,,,,data science
1,1,Data Scientist Intern — ML & experimentation,responsibility,"machine learning, data analysis, experimentati...","Python, SQL, PyTorch, TensorFlow, Azure","experimentation, statistical analysis, model e...","PyTorch, TensorFlow, Azure",,AI products
2,2,Product-focused Data Scientist / ML Engineer,responsibility,"collaboration, data analysis, machine learning...","Python, SQL, PyTorch, TensorFlow, Azure, Power...","collaboration, statistical analysis, modeling,...","PyTorch, TensorFlow, Azure, data visualization...",,AI products
3,3,ML/Data Exploration Internship,responsibility,"data exploration, statistical analysis, machin...","Python, SQL, pandas, scikit-learn, Jupyter","data exploration, statistical analysis, patter...",Jupyter,,AI products
4,4,Learn production data science and product anal...,responsibility,"data science, evaluation, data-driven decision...","Python, SQL, Jupyter, Git","evaluation, data-driven decision making, produ...",Jupyter,,product analytics
5,5,Communicate data insights to product stakeholders,responsibility,"communication, data analysis, data visualizati...","Python, SQL, Tableau, Looker, Jupyter, AWS, Py...","statistics, evaluation, stakeholder management...","Tableau, Looker, Jupyter, AWS, PyTorch, Tensor...",,AI products
6,6,Microsoft internship (general),responsibility,,,,,,general internship
7,7,General responsibilities,responsibility,,,,,,
8,8,ML & statistical analysis for product problems,responsibility,"statistical analysis, machine learning, data e...","Python, SQL, pandas, scikit-learn, PyTorch, Te...","statistical analysis, data exploration, featur...","PyTorch, TensorFlow, AWS",,AI products
9,9,Analyze structured & unstructured data,responsibility,"data analysis, exploratory data analysis, stat...","Python, SQL, Pandas, Jupyter, Spark, AWS, Tens...","exploratory data analysis, statistical analysi...","Jupyter, Spark, AWS, TensorFlow, PyTorch",,data analytics


===== REWRITE SUGGESTIONS =====


,jd_req_id,jd_requirement,resume_strength,why,rewritten_bullets
0,0,general — Data Scientist Intern – Microsoft,medium,"Provides direct, relevant experience in machin...",• Developed and optimized machine learning mod...
1,1,"general — At Microsoft, Data Scientist Interns...",strong,"Experience shows direct, product-focused machi...",• Developed and optimized AI-driven threat det...
2,2,"general — You will collaborate with engineers,...",medium,The resume shows direct experience analyzing d...,• Developed and optimized AI-driven threat det...
3,3,"general — During the internship, you will expl...",strong,Resume shows direct experience exploring real ...,• Developed and optimized AI-driven threat det...
4,4,general — You will learn how data science deci...,strong,Experience shows hands-on work converting anal...,"• Optimized AI-driven threat detection models,..."
5,5,general — You will also communicate your findi...,medium,"The resume shows relevant, transferable eviden...",• Developed a Streamlit UI with Matplotlib vis...
6,6,general — Microsoft internships are designed t...,strong,The candidate has hands-on internship and proj...,• Developed and optimized AI-driven threat det...
7,7,general — Responsibilities,strong,The resume shows direct experience developing ...,• Developed and optimized AI-driven threat det...
8,8,"general — Apply statistical analysis, machine ...",strong,Resume shows multiple applied ML and data expl...,• Developed and optimized AI-driven threat det...
9,9,general — Work with structured and unstructure...,strong,Resume shows direct experience processing both...,• Developed and optimized AI-driven threat det...


In [29]:
CANONICAL_SKILL_MAP = {
    "machine learning": [
        "ml", "machine learning", "predictive modeling", "supervised learning",
        "unsupervised learning", "classification", "regression"
    ],
    "deep learning": [
        "deep learning", "neural networks", "dnn", "cnn", "rnn", "lstm"
    ],
    "nlp": [
        "nlp", "natural language processing", "text processing", "text analytics",
        "language modeling"
    ],
    "llm": [
        "llm", "large language model", "large language models", "foundation model",
        "generative ai", "genai"
    ],
    "rag": [
        "rag", "retrieval augmented generation", "retrieval-augmented generation"
    ],
    "transformers": [
        "transformer", "transformers", "attention mechanism", "self-attention"
    ],
    "computer vision": [
        "computer vision", "cv", "image processing", "object detection",
        "image classification"
    ],
    "python": [
        "python"
    ],
    "sql": [
        "sql", "postgresql", "mysql", "sqlite", "structured query language"
    ],
    "pandas": [
        "pandas"
    ],
    "numpy": [
        "numpy"
    ],
    "scikit-learn": [
        "scikit-learn", "sklearn"
    ],
    "tensorflow": [
        "tensorflow", "tf", "keras"
    ],
    "pytorch": [
        "pytorch", "torch", "pytorch lightning"
    ],
    "langchain": [
        "langchain"
    ],
    "vector databases": [
        "vector db", "vector database", "vector databases", "faiss", "chroma",
        "pinecone", "weaviate", "milvus"
    ],
    "embeddings": [
        "embedding", "embeddings", "sentence embeddings", "text embeddings"
    ],
    "fine-tuning": [
        "fine tuning", "fine-tuning", "sft", "supervised fine-tuning"
    ],
    "prompt engineering": [
        "prompt engineering", "prompt design", "prompting"
    ],
    "evaluation": [
        "evaluation", "model evaluation", "benchmarking", "metrics",
        "precision", "recall", "f1 score", "accuracy"
    ],
    "deployment": [
        "deployment", "deploying", "production deployment", "serving",
        "model serving", "production"
    ],
    "mlops": [
        "mlops", "machine learning ops", "machine learning operations",
        "model monitoring", "experiment tracking"
    ],
    "aws": [
        "aws", "amazon web services", "sagemaker", "ec2", "lambda", "bedrock"
    ],
    "gcp": [
        "gcp", "google cloud", "vertex ai", "bigquery"
    ],
    "azure": [
        "azure", "microsoft azure", "azure ml", "azure machine learning"
    ],
    "docker": [
        "docker", "containerization", "containers"
    ],
    "kubernetes": [
        "kubernetes", "k8s"
    ],
    "api development": [
        "api", "rest api", "fastapi", "flask", "backend api"
    ],
    "data visualization": [
        "matplotlib", "seaborn", "plotly", "dashboard", "data visualization"
    ],
    "experimentation": [
        "a/b testing", "ab testing", "experimentation", "hypothesis testing"
    ],
    "statistics": [
        "statistics", "statistical modeling", "probability", "hypothesis testing"
    ]
}

In [30]:
ALIAS_TO_CANONICAL = {}

for canonical, aliases in CANONICAL_SKILL_MAP.items():
    for alias in aliases:
        ALIAS_TO_CANONICAL[alias.lower().strip()] = canonical

In [31]:
def clean_skill_text(text: str) -> str:
    text = text.lower().strip()
    text = text.replace("&", " and ")
    text = re.sub(r"[^a-z0-9+#./ -]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def canonicalize_skill(skill: str) -> str:
    s = clean_skill_text(skill)

    if s in ALIAS_TO_CANONICAL:
        return ALIAS_TO_CANONICAL[s]

    for alias, canonical in ALIAS_TO_CANONICAL.items():
        if s == alias:
            return canonical

    for alias, canonical in ALIAS_TO_CANONICAL.items():
        if alias in s or s in alias:
            return canonical

    return s


def extract_canonical_skills_from_text(text: str) -> set:
    cleaned = clean_skill_text(text)
    found = set()

    for alias, canonical in ALIAS_TO_CANONICAL.items():
        pattern = r"\b" + re.escape(alias) + r"\b"
        if re.search(pattern, cleaned):
            found.add(canonical)

    return found

In [32]:
def normalize_structured_jd_df(structured_jd_df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for _, row in structured_jd_df.iterrows():
        raw_skills = row.get("skills", []) or []
        raw_tools = row.get("tools", []) or []

        norm_skills = sorted(set(canonicalize_skill(x) for x in raw_skills if str(x).strip()))
        norm_tools = sorted(set(canonicalize_skill(x) for x in raw_tools if str(x).strip()))

        rows.append({
            "jd_req_id": row["jd_req_id"],
            "requirement_text": row.get("requirement_text", ""),
            "short_label": row.get("short_label", ""),
            "category": row.get("category", ""),
            "skills_raw": raw_skills,
            "tools_raw": raw_tools,
            "skills_canonical": norm_skills,
            "tools_canonical": norm_tools,
            "years_required": row.get("years_required", ""),
            "domain": row.get("domain", ""),
            "must_have": row.get("must_have", True)
        })

    return pd.DataFrame(rows)

In [33]:
def extract_resume_skill_inventory(resume_chunks: List[Document]) -> pd.DataFrame:
    rows = []

    for doc in resume_chunks:
        text = normalize_text(doc.page_content)
        found_skills = sorted(extract_canonical_skills_from_text(text))

        rows.append({
            "page": doc.metadata.get("page"),
            "section": doc.metadata.get("section"),
            "source_file": doc.metadata.get("source_file"),
            "chunk_id": doc.metadata.get("chunk_id"),
            "text": text,
            "canonical_skills_found": found_skills
        })

    return pd.DataFrame(rows)

In [34]:
def build_resume_skill_set(resume_skill_inventory_df: pd.DataFrame) -> set:
    all_skills = set()

    for skills in resume_skill_inventory_df["canonical_skills_found"]:
        if isinstance(skills, list):
            all_skills.update(skills)

    return all_skills

In [35]:
def compute_gap_analysis_canonical(
    structured_jd_norm_df: pd.DataFrame,
    resume_skill_set: set
) -> pd.DataFrame:
    rows = []

    for _, row in structured_jd_norm_df.iterrows():
        jd_skills = set(row.get("skills_canonical", []) or [])
        jd_tools = set(row.get("tools_canonical", []) or [])

        required_items = sorted(jd_skills | jd_tools)
        matched_items = sorted([x for x in required_items if x in resume_skill_set])
        missing_items = sorted([x for x in required_items if x not in resume_skill_set])

        total_required = len(required_items)
        matched_count = len(matched_items)
        missing_count = len(missing_items)

        if total_required == 0:
            gap_score = 0.0
            coverage_ratio = 1.0
        else:
            coverage_ratio = matched_count / total_required
            gap_score = 1 - coverage_ratio

        rows.append({
            "jd_req_id": row["jd_req_id"],
            "short_label": row.get("short_label", ""),
            "category": row.get("category", ""),
            "required_items": required_items,
            "matched_items": matched_items,
            "missing_items": missing_items,
            "matched_count": matched_count,
            "missing_count": missing_count,
            "coverage_ratio": round(coverage_ratio, 3),
            "gap_score": round(gap_score, 3),
            "years_required": row.get("years_required", ""),
            "domain": row.get("domain", ""),
            "must_have": row.get("must_have", True)
        })

    return pd.DataFrame(rows)

In [36]:
def assign_gap_severity(gap_df: pd.DataFrame) -> pd.DataFrame:
    df = gap_df.copy()

    severities = []
    priorities = []

    for _, row in df.iterrows():
        gap_score = row["gap_score"]
        must_have = row.get("must_have", True)
        missing_count = row.get("missing_count", 0)

        if missing_count == 0:
            severity = "none"
            priority = "low"
        elif must_have and gap_score >= 0.75:
            severity = "high"
            priority = "high"
        elif must_have and gap_score >= 0.4:
            severity = "medium"
            priority = "high"
        elif gap_score >= 0.75:
            severity = "high"
            priority = "medium"
        elif gap_score >= 0.4:
            severity = "medium"
            priority = "medium"
        else:
            severity = "low"
            priority = "low"

        severities.append(severity)
        priorities.append(priority)

    df["gap_severity"] = severities
    df["priority"] = priorities

    return df

In [37]:
def build_skill_evidence_map(resume_chunks: List[Document]) -> dict:
    evidence_map = {}

    for doc in resume_chunks:
        text = normalize_text(doc.page_content)
        found = extract_canonical_skills_from_text(text)

        for skill in found:
            evidence_map.setdefault(skill, []).append({
                "page": doc.metadata.get("page"),
                "section": doc.metadata.get("section"),
                "source_file": doc.metadata.get("source_file"),
                "chunk_id": doc.metadata.get("chunk_id"),
                "evidence": snippet(text, 220)
            })

    return evidence_map

In [38]:
def skill_evidence_map_to_df(skill_evidence_map: dict) -> pd.DataFrame:
    rows = []

    for skill, evidences in skill_evidence_map.items():
        for ev in evidences:
            rows.append({
                "canonical_skill": skill,
                "page": ev["page"],
                "section": ev["section"],
                "source_file": ev["source_file"],
                "chunk_id": ev["chunk_id"],
                "evidence": ev["evidence"]
            })

    return pd.DataFrame(rows)

In [39]:
def build_skill_gap_summary(gap_df: pd.DataFrame) -> pd.DataFrame:
    all_required = Counter()
    all_missing = Counter()
    all_matched = Counter()

    for _, row in gap_df.iterrows():
        for item in row.get("required_items", []) or []:
            all_required[item] += 1
        for item in row.get("missing_items", []) or []:
            all_missing[item] += 1
        for item in row.get("matched_items", []) or []:
            all_matched[item] += 1

    skills = sorted(set(all_required) | set(all_missing) | set(all_matched))

    rows = []
    for skill in skills:
        rows.append({
            "canonical_skill": skill,
            "times_required_in_jd": all_required.get(skill, 0),
            "times_matched_in_resume": all_matched.get(skill, 0),
            "times_missing": all_missing.get(skill, 0)
        })

    return pd.DataFrame(rows).sort_values(
        ["times_missing", "times_required_in_jd"],
        ascending=[False, False]
    )

In [40]:
structured_jd_norm_df = normalize_structured_jd_df(structured_jd_df)

resume_skill_inventory_df = extract_resume_skill_inventory(resume_chunks)
resume_skill_set = build_resume_skill_set(resume_skill_inventory_df)

gap_df_canonical = compute_gap_analysis_canonical(
    structured_jd_norm_df=structured_jd_norm_df,
    resume_skill_set=resume_skill_set
)

gap_df_canonical = assign_gap_severity(gap_df_canonical)

skill_evidence_map = build_skill_evidence_map(resume_chunks)
skill_evidence_df = skill_evidence_map_to_df(skill_evidence_map)

skill_gap_summary_df = build_skill_gap_summary(gap_df_canonical)

print("===== NORMALIZED JD =====")
display(structured_jd_norm_df.head(10))

print("===== RESUME SKILL INVENTORY =====")
display(resume_skill_inventory_df.head(10))

print("===== CANONICAL GAP ANALYSIS =====")
display(gap_df_canonical.head(15))

print("===== SKILL GAP SUMMARY =====")
display(skill_gap_summary_df.head(15))

===== NORMALIZED JD =====


,jd_req_id,requirement_text,short_label,category,skills_raw,tools_raw,skills_canonical,tools_canonical,years_required,domain,must_have
0,0,general — Data Scientist Intern – Microsoft,Data Scientist Intern,other,"[data analysis, machine learning, statistics, ...","[Python, SQL, R, scikit-learn, PyTorch, Tensor...","[communication, data analysis, data visualizat...","[azure, excel, git, machine learning, python, ...",,data science,True
1,1,"general — At Microsoft, Data Scientist Interns...",Data Scientist Intern — ML & experimentation,responsibility,"[machine learning, data analysis, experimentat...",[],"[data analysis, experimentation, feature engin...",[],,AI products,True
2,2,"general — You will collaborate with engineers,...",Analyze data and build models to improve Micro...,responsibility,"[data analysis, machine learning, statistical ...","[Python, SQL, PyTorch, TensorFlow, Azure, Jupy...","[cross-functional collaboration, data analysis...","[azure, git, jupyter, python, pytorch, sql, te...",,AI products,True
3,3,"general — During the internship, you will expl...",Data exploration & ML (internship),responsibility,"[data exploration, statistical analysis, machi...","[Python, SQL, pandas, scikit-learn, Jupyter, m...","[data exploration, data visualization, evaluat...","[data visualization, jupyter, pandas, python, ...",,data science,True
4,4,general — You will learn how data science deci...,Production data science & product analytics,responsibility,"[data science, analytics, data-driven decision...",[],"[communication, data science, data-driven deci...",[],,AI products,True
5,5,general — You will also communicate your findi...,Stakeholder communication & product guidance,responsibility,"[data analysis, communication, stakeholder man...",[],"[communication, cross-functional collaboration...",[],,AI products,True
6,6,general — Microsoft internships are designed t...,Microsoft internship (general),responsibility,"[data science, software engineering, problem s...",[],"[collaboration, data science, machine learning...",[],,software & data science,True
7,7,general — Responsibilities,General responsibilities,responsibility,[],[],[],[],,,True
8,8,"general — Apply statistical analysis, machine ...",ML & statistical analysis for product problems,skills,"[statistical analysis, machine learning, data ...","[Python, SQL, Pandas, scikit-learn, Jupyter]","[data analysis, data exploration, data visuali...","[jupyter, pandas, python, scikit-learn, sql]",,AI products,True
9,9,general — Work with structured and unstructure...,Analyze structured & unstructured data,responsibility,"[data analysis, data mining, exploratory data ...","[Python, SQL, AWS, PyTorch, TensorFlow, Pandas...","[data analysis, data mining, evaluation, explo...","[aws, pandas, python, pytorch, spark, sql, ten...",,data analytics,True


===== RESUME SKILL INVENTORY =====


,page,section,source_file,chunk_id,text,canonical_skills_found
0,0,unknown,Devanshi_Faldu_s_Resume (2).pdf,0,Devanshi Faldu 6351478229 • devanshifaldu@gmai...,[]
1,0,education,Devanshi_Faldu_s_Resume (2).pdf,1,DAYANAND SAGAR UNIVERSITY | Masters of Science...,"[data visualization, python]"
2,0,experience,Devanshi_Faldu_s_Resume (2).pdf,2,AI INTERN Nexeo Security September 2024 - Marc...,"[evaluation, machine learning, nlp, python]"
3,0,projects,Devanshi_Faldu_s_Resume (2).pdf,3,MOVIE RECOMMENDATION | Similar movie suggestio...,"[data visualization, numpy, pandas, python, sc..."
4,0,experience,Devanshi_Faldu_s_Resume (2).pdf,4,•Implement 3 combined feature vectors to compu...,[]
5,0,skills,Devanshi_Faldu_s_Resume (2).pdf,5,TECHNICALS | Power BI | Data Visualisation | D...,"[machine learning, nlp, python, sql]"
6,0,summary,Devanshi_Faldu_s_Resume (2).pdf,6,LINKEDIN Linkedin/in/devanshi-faldu GITHUB Git...,[]
7,0,summary,Devanshi_Faldu_s_Resume (2).pdf,7,GEEKSFORGEEKS geeksforgeeks.org/user/devanshbq8l,[]


===== CANONICAL GAP ANALYSIS =====


,jd_req_id,short_label,category,required_items,matched_items,missing_items,matched_count,missing_count,coverage_ratio,gap_score,years_required,domain,must_have,gap_severity,priority
0,0,Data Scientist Intern,other,"[azure, communication, data analysis, data vis...","[data visualization, machine learning, python,...","[azure, communication, data analysis, excel, e...",5,10,0.333,0.667,,data science,True,medium,high
1,1,Data Scientist Intern — ML & experimentation,responsibility,"[data analysis, experimentation, feature engin...",[machine learning],"[data analysis, experimentation, feature engin...",1,5,0.167,0.833,,AI products,True,high,high
2,2,Analyze data and build models to improve Micro...,responsibility,"[azure, cross-functional collaboration, data a...","[data visualization, evaluation, machine learn...","[azure, cross-functional collaboration, data a...",5,10,0.333,0.667,,AI products,True,medium,high
3,3,Data exploration & ML (internship),responsibility,"[data exploration, data visualization, evaluat...","[data visualization, evaluation, machine learn...","[data exploration, feature engineering, jupyte...",7,5,0.583,0.417,,data science,True,medium,high
4,4,Production data science & product analytics,responsibility,"[communication, data science, data-driven deci...","[evaluation, nlp]","[communication, data science, data-driven deci...",2,6,0.250,0.750,,AI products,True,high,high
5,5,Stakeholder communication & product guidance,responsibility,"[communication, cross-functional collaboration...",[data visualization],"[communication, cross-functional collaboration...",1,7,0.125,0.875,,AI products,True,high,high
6,6,Microsoft internship (general),responsibility,"[collaboration, data science, machine learning...",[machine learning],"[collaboration, data science, problem solving,...",1,4,0.200,0.800,,software & data science,True,high,high
7,7,General responsibilities,responsibility,[],[],[],0,0,1.000,0.000,,,True,none,low
8,8,ML & statistical analysis for product problems,skills,"[data analysis, data exploration, data visuali...","[data visualization, evaluation, machine learn...","[data analysis, data exploration, feature engi...",7,5,0.583,0.417,,AI products,True,medium,high
9,9,Analyze structured & unstructured data,responsibility,"[aws, data analysis, data mining, evaluation, ...","[evaluation, machine learning, nlp, pandas, py...","[aws, data analysis, data mining, exploratory ...",6,9,0.400,0.600,,data analytics,True,medium,high


===== SKILL GAP SUMMARY =====


,canonical_skill,times_required_in_jd,times_matched_in_resume,times_missing
9,data analysis,12,0,12
51,statistical analysis,9,0,9
30,jupyter,6,0,6
6,communication,5,0,5
25,feature engineering,5,0,5
23,experimentation,4,0,4
40,product analytics,4,0,4
54,tensorflow,4,0,4
2,azure,3,0,3
7,cross-functional collaboration,3,0,3


In [41]:
final_master_canonical_df = (
    summary_df
    .merge(
        structured_jd_norm_df[[
            "jd_req_id", "short_label", "category",
            "skills_canonical", "tools_canonical",
            "years_required", "domain", "must_have"
        ]],
        on="jd_req_id",
        how="left"
    )
    .merge(
        gap_df_canonical[[
            "jd_req_id", "required_items", "matched_items", "missing_items",
            "coverage_ratio", "gap_score", "gap_severity", "priority"
        ]],
        on="jd_req_id",
        how="left"
    )
    .merge(
        rewrite_df[[
            "jd_req_id", "resume_strength", "why", "rewritten_bullets"
        ]],
        on="jd_req_id",
        how="left"
    )
)

print("===== FINAL MASTER CANONICAL OUTPUT =====")
display(final_master_canonical_df.head(20))

===== FINAL MASTER CANONICAL OUTPUT =====


,jd_req_id,jd_requirement,best_final_score,resume_page,resume_section,source_file,evidence,short_label,category,skills_canonical,...,required_items,matched_items,missing_items,coverage_ratio,gap_score,gap_severity,priority,resume_strength,why,rewritten_bullets
0,16,"general — Experience with Python, R, or simila...",0.6344,0,skills,Devanshi_Faldu_s_Resume (2).pdf,TECHNICALS | Power BI | Data Visualisation | D...,Python/R for data analysis,tools,"[data analysis, statistical analysis]",...,"[data analysis, machine learning, programming ...","[machine learning, python]","[data analysis, programming languages, statist...",0.400,0.600,medium,high,strong,"Resume shows repeated, concrete use of Python ...",• Developed Python-based AI threat detection a...
1,17,"general — Familiarity with statistics, machine...",0.4716,0,experience,Devanshi_Faldu_s_Resume (2).pdf,AI INTERN Nexeo Security September 2024 - Marc...,Familiarity with statistics & ML,skills,"[data analysis, data analysis techniques, mach...",...,"[data analysis, data analysis techniques, mach...",[machine learning],"[data analysis, data analysis techniques, stat...",0.200,0.800,high,high,strong,Resume demonstrates direct experience applying...,• Developed and optimized AI-driven threat det...
2,0,general — Data Scientist Intern – Microsoft,0.4637,0,experience,Devanshi_Faldu_s_Resume (2).pdf,AI INTERN Nexeo Security September 2024 - Marc...,Data Scientist Intern,other,"[communication, data analysis, data visualizat...",...,"[azure, communication, data analysis, data vis...","[data visualization, machine learning, python,...","[azure, communication, data analysis, excel, e...",0.333,0.667,medium,high,medium,"Provides direct, relevant experience in machin...",• Developed and optimized machine learning mod...
3,8,"general — Apply statistical analysis, machine ...",0.4392,0,experience,Devanshi_Faldu_s_Resume (2).pdf,AI INTERN Nexeo Security September 2024 - Marc...,ML & statistical analysis for product problems,skills,"[data analysis, data exploration, data visuali...",...,"[data analysis, data exploration, data visuali...","[data visualization, evaluation, machine learn...","[data analysis, data exploration, feature engi...",0.583,0.417,medium,high,strong,Resume shows multiple applied ML and data expl...,• Developed and optimized AI-driven threat det...
4,1,"general — At Microsoft, Data Scientist Interns...",0.4116,0,experience,Devanshi_Faldu_s_Resume (2).pdf,AI INTERN Nexeo Security September 2024 - Marc...,Data Scientist Intern — ML & experimentation,responsibility,"[data analysis, experimentation, feature engin...",...,"[data analysis, experimentation, feature engin...",[machine learning],"[data analysis, experimentation, feature engin...",0.167,0.833,high,high,strong,"Experience shows direct, product-focused machi...",• Developed and optimized AI-driven threat det...
5,10,general — Design and analyze experiments to ev...,0.3821,0,projects,Devanshi_Faldu_s_Resume (2).pdf,MOVIE RECOMMENDATION | Similar movie suggestio...,Experiment design & analysis,responsibility,"[causal inference, data analysis, evaluation, ...",...,"[causal inference, data analysis, evaluation, ...","[evaluation, machine learning, python, sql]","[causal inference, data analysis, experimental...",0.333,0.667,medium,high,medium,Evidence shows hands-on experience designing e...,• Designed evaluation components for a content...
6,3,"general — During the internship, you will expl...",0.3805,0,experience,Devanshi_Faldu_s_Resume (2).pdf,AI INTERN Nexeo Security September 2024 - Marc...,Data exploration & ML (internship),responsibility,"[data exploration, data visualization, evaluat...",...,"[data exploration, data visualization, evaluat...","[data visualization, evaluation, machine learn...","[data exploration, feature engineering, jupyte...",0.583,0.417,medium,high,strong,Resume shows direct experience exploring real ...,• Developed and optimized AI-driven threat det...
7,9,general — Work with structured and un